In [ ]:
# @title License
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Aerial and Satellite Models: Region Search with BigQuery Vector Search

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fearth_ai_remote_sensing%2Fnotebooks%2Fregion_search_bigquery%2Fearth_ai_region_search_bigquery.ipynb?utm_source=earth_ai_remote_sensing_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/earth_ai_remote_sensing/notebooks/region_search_bigquery/earth_ai_region_search_bigquery.ipynb&utm_source=earth_ai_remote_sensing_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/googlemaps-samples/insights-samples/main/earth_ai_remote_sensing/notebooks/region_search_bigquery/earth_ai_region_search_bigquery.ipynb&utm_source=earth_ai_remote_sensing_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Workbench logo"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/earth_ai_remote_sensing/notebooks/region_search_bigquery/earth_ai_region_search_bigquery.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>
<br clear="all"/>

## Overview

### Overall Goal
This notebook provides a production-ready blueprint for **metro-scale overhead imagery discovery and opportunity analysis** using **Google Maps Platform Aerial and Satellite Models** on **Agent Platform Model Garden** (part of the **Google Earth AI Remote Sensing Foundation Models** effort) together with **BigQuery Vector Search**:
1. **Aerial and Satellite Models (`RS-SigLIP2`) on Agent Platform Model Garden** — deployed via the official [`google-research/remote-sensing`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py) `remote_sensing.vertex_ai.utils` module — encode overhead aerial/satellite imagery tiles and natural-language text prompts into a shared, L2-normalized **`1152-D` vector space**.
2. **BigQuery (`VECTOR_SEARCH` & `ML.DISTANCE`)** indexes tens of thousands of regional orthoimagery tiles and executes sub-second **Text-to-Tile Semantic Search**, **Tile-to-Tile Visual Similarity**, and **Opportunity Finder (Vector Algebra)** queries in standard SQL (`$0/hr` idle GPU cost).
3. **Interactive Map Visualization (`Folium` + Google Maps 2D Tiles)** overlays the discovered `307 m × 307 m` tile footprints onto an interactive map for visual inspection.

---

### Architecture & Division of Labor

| Stage | Component | What It Does | Frequency & Cost Profile |
| :--- | :--- | :--- | :--- |
| **1. Public-Domain Tile Grid** | **USDA / USGS NAIP ImageServer** | Slices an Area of Interest (AOI) into geo-referenced `512×512 px` (`0.6 m` GSD, `~307 m × 307 m`) aerial orthoimagery tiles (`17 U.S.C. § 105`). | **One-time / Periodic** refresh. |
| **2. Feature Extraction** | **Aerial and Satellite Models (`RS-SigLIP2`)** *(Agent Platform Model Garden)* | Uses `remote_sensing.vertex_ai.utils` (`create_model` / `deploy_model` / `batch_predict`) to encode overhead tiles (`IMAGE_ONLY`) and text prompts (`TEXT_ONLY`) into L2-normalized `ARRAY<FLOAT64>` (`1152-D`) vectors. | **One-time GPU job** for tiles; **< 50 ms** for new text prompts. |
| **3. Spatial & Vector Indexing** | **BigQuery (`GEOGRAPHY` + `VECTOR INDEX`)** | Stores tile bounding boxes, `1152-D` image vectors, and pre-encoded text vectors in native BigQuery tables. | **Serverless Storage** (`$0.02 / GB / month`). |
| **4. Metro-Scale Retrieval** | **BigQuery `ML.DISTANCE` & `VECTOR_SEARCH`** | Computes cosine similarity (`1 - ML.DISTANCE(img, txt, 'COSINE')`), regional z-scores (`+σ`), and composite vector algebra (`pos - α·neg`). | **Pure SQL** (`$0/hr` idle GPU cost once indexed). |

---

### Terms of Use & Public-Domain Data Provenance
- **Overhead Imagery for Model Inference (`17 U.S.C. § 105`)**: All overhead imagery tiles passed into the Aerial and Satellite Model (`RS-SigLIP2`) in this notebook are sourced from the **U.S. Department of Agriculture (USDA) / USGS National Agriculture Imagery Program (NAIP)** (`0.6 m` ground sample distance, `512×512 px`), which is a work of the U.S. Federal Government in the **public domain**.
- **Google Maps Platform Terms of Service (Section 3.2.3)**: Google Maps 2D Satellite/Hybrid tiles are used strictly as an **interactive visual basemap overlay** via the official Map Tiles API in Step 6 and are **never** downloaded, scraped, cached, or passed as input into any ML model.

---

### How to Use This Notebook
1. **Run Step 1 (`Setup & Authentication`)** to install dependencies, authenticate with Google Cloud, and clone the official [`google-research/remote-sensing`](https://github.com/google-research/remote-sensing) repository (`remote_sensing.vertex_ai.utils`).
2. **Configure Step 2 (`Configuration & All Notebook Parameters`)** — **all user-configurable `# @param` parameters for the entire notebook are consolidated in that single cell** at the top of the notebook.
3. **Run Steps 3–7 sequentially** to deploy or attach to an **Aerial and Satellite Models (`RS-SigLIP2`)** endpoint via `remote_sensing.vertex_ai.utils` (or query existing BigQuery tables directly), inspect/populate the `1152-D` vector tables, run **Text-to-Tile Search**, **Tile-to-Tile Visual Similarity**, **Opportunity Finder Vector Algebra**, and **Interactive Map Visualization**.

## Step 1: Setup & Authentication

First, install the required Google Cloud, Agent Platform, BigQuery, and visualization libraries, clone the official [`google-research/remote-sensing`](https://github.com/google-research/remote-sensing) repository (which provides [`remote_sensing.vertex_ai.utils`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py)), and authenticate your environment:

- **Colab Enterprise / Workbench / BigQuery Studio**: Uses your attached runtime credentials automatically via `google.auth.default()`.
- **Public Google Colab**: Prompts for browser authentication via `google.colab.auth.authenticate_user()` and automatically reads `GCP_PROJECT_ID` and `GMP_API_KEY` from `google.colab.userdata` secrets if configured.

In [ ]:
# @title 1. Install Dependencies, Clone `google-research/remote-sensing` & Authenticate
!pip install --quiet --upgrade google-cloud-aiplatform google-cloud-bigquery google-cloud-storage db-dtypes pandas matplotlib pillow requests folium

import io
import json
import math
import os
import sys
import uuid
import warnings
from typing import Any, Dict, List, Optional, Tuple

import folium
from google.cloud import aiplatform
from google.cloud import bigquery
from google.cloud import storage
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 1000)

# Clone official Earth AI Remote Sensing repository for remote_sensing.vertex_ai.utils
if not os.path.exists("remote-sensing"):
    !git clone --quiet https://github.com/google-research/remote-sensing.git
repo_path = os.path.abspath("remote-sensing")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from remote_sensing.vertex_ai import utils

# Dual-mode authentication (Colab userdata / interactive auth + Enterprise ADC fallback)
_COLAB_SECRET_PROJECT = ""
_COLAB_SECRET_GMP_KEY = ""
try:
    from google.colab import auth as colab_auth
    from google.colab import userdata

    colab_auth.authenticate_user()
    try:
        _COLAB_SECRET_PROJECT = userdata.get("GCP_PROJECT_ID") or ""
    except Exception:
        pass
    try:
        _COLAB_SECRET_GMP_KEY = userdata.get("GMP_API_KEY") or ""
    except Exception:
        pass
    print("✅ Authenticated via Google Colab user session.")
except ImportError:
    import google.auth

    _, _adc_project = google.auth.default()
    _COLAB_SECRET_PROJECT = _adc_project or ""
    print("✅ Authenticated via Application Default Credentials (Colab Enterprise / Workbench).")

## Step 2: Configuration & All Notebook Parameters

All configurable parameters for the entire notebook are consolidated in the single form below so you can customize your run in one place before executing the workflow:

| Parameter Group | Key Parameters | Purpose |
| :--- | :--- | :--- |
| **1. Google Cloud & BigQuery** | `PROJECT_ID`, `REGION`, `BQ_DATASET`, `TILES_TABLE_NAME`, `QUERIES_TABLE_NAME`, `GMP_API_KEY` | Selects your GCP project, BigQuery dataset/tables (`region_tiles_la` or `region_tiles_sf`), and optional Google Maps 2D Tile API key. |
| **2. AOI & Public-Domain NAIP Grid** | `AOI_NAME`, `AOI_MIN_LAT`, `AOI_MIN_LNG`, `AOI_MAX_LAT`, `AOI_MAX_LNG`, `MAX_DEMO_TILES`, `FETCH_LIVE_NAIP_TILES` | Defines the bounding box for live USGS NAIP (`0.6 m` GSD, `512×512 px`) tile fetching when building a new table from scratch. |
| **3. Agent Platform Model Garden (`remote_sensing.vertex_ai.utils`)** | `DEPLOY_LIVE_ENDPOINT`, `EXISTING_ENDPOINT_ID`, `MODEL_TYPE`, `MODEL_MODE`, `ACCELERATOR`, `USE_DEDICATED_ENDPOINT`, `MIN_REPLICA_COUNT`, `MAX_REPLICA_COUNT`, `RUN_BATCH_JOB_EXAMPLE` | Configures `utils.create_model(...)` and `utils.deploy_model(...)` from [`remote_sensing/vertex_ai/utils.py`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py) for the **RS-SigLIP2** (`SIGLIP2_512_RC1` / `SIGLIP2_224_RC1`) Aerial and Satellite Model, or runs immediately against pre-computed BigQuery tables (`DEPLOY_LIVE_ENDPOINT = False`). |
| **4. Search & Vector Algebra** | `SEARCH_QUERY_ID`, `SEED_TILE_ID`, `POSITIVE_QUERY_ID`, `NEGATIVE_QUERY_ID`, `NEGATIVE_WEIGHT`, `TOP_K_RESULTS` | Configures the prompts, seed tile, and composite vector algebra weights for **Step 5 (5A, 5B, and 5C)**. |
| **5. Resource Cleanup** | `UNDEPLOY_ENDPOINT_ON_FINISH`, `DELETE_ENDPOINT_AND_MODEL` | Controls automatic GPU endpoint teardown in **Step 7**. |

In [ ]:
# @title 2. Configuration & All Notebook Parameters

# @markdown ### 1. Google Cloud, BigQuery & Maps Basemap Settings
PROJECT_ID = ""  # @param {type:"string"}
REGION = "us-central1"  # @param ["us-central1", "us-east4", "us-west1", "europe-west4"]
BQ_DATASET = "rsfm_demo"  # @param {type:"string"}
TILES_TABLE_NAME = "region_tiles_la"  # @param ["region_tiles_la", "region_tiles_sf"]
QUERIES_TABLE_NAME = "region_queries_la"  # @param ["region_queries_la", "region_queries_sf"]
GMP_API_KEY = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 2. Area of Interest (AOI) & Public-Domain USGS NAIP Grid Settings
# @markdown Used when `FETCH_LIVE_NAIP_TILES = True` to fetch live public-domain `512x512` (`0.6m` GSD) USDA/USGS NAIP orthoimagery tiles.
AOI_NAME = "Port of Los Angeles / San Pedro"  # @param {type:"string"}
AOI_MIN_LAT = 33.7250  # @param {type:"number"}
AOI_MIN_LNG = -118.2800  # @param {type:"number"}
AOI_MAX_LAT = 33.7550  # @param {type:"number"}
AOI_MAX_LNG = -118.2400  # @param {type:"number"}
TILE_SIZE_PX = 512  # @param [224, 512] {type:"raw"}
MAX_DEMO_TILES = 16  # @param {type:"slider", min:4, max:64, step:4}
FETCH_LIVE_NAIP_TILES = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 3. Aerial and Satellite Models — Agent Platform Model Garden Settings (`remote_sensing.vertex_ai.utils`)
# @markdown Set `DEPLOY_LIVE_ENDPOINT = False` (default) to immediately run BigQuery `VECTOR_SEARCH` over pre-computed `1152-D` `RS-SigLIP2` embeddings without waiting ~10 minutes for a GPU endpoint deployment. Set to `True` to deploy a live endpoint via `utils.create_model` & `utils.deploy_model`, or attach to `EXISTING_ENDPOINT_ID`.
DEPLOY_LIVE_ENDPOINT = False  # @param {type:"boolean"}
EXISTING_ENDPOINT_ID = ""  # @param {type:"string"}
DISPLAY_NAME = "aerial-satellite-siglip2"  # @param {type:"string"}
MODEL_TYPE = "SIGLIP2_512_RC1"  # @param ["MAMMUT", "SIGLIP2_224_RC1", "SIGLIP2_512_RC1"]
MODEL_MODE = "COMBINED"  # @param ["IMAGE_ONLY", "TEXT_ONLY", "COMBINED"]
ACCELERATOR = "NVIDIA_L4"  # @param ["CPU", "NVIDIA_L4", "NVIDIA_TESLA_A100", "NVIDIA_A100_80GB"]
USE_DEDICATED_ENDPOINT = True  # @param {type:"boolean"}
SERVICE_ACCOUNT = ""  # @param {type:"string"}
MIN_REPLICA_COUNT = 1  # @param {type:"integer"}
MAX_REPLICA_COUNT = 1  # @param {type:"integer"}
RUN_BATCH_JOB_EXAMPLE = False  # @param {type:"boolean"}
GCS_BUCKET_NAME = ""  # @param {type:"string"}
INPUT_SOURCE_FORMAT = "jsonl"  # @param ["file-list", "jsonl"]
BATCH_SIZE = 4  # @param {type:"integer"}

# @markdown ---
# @markdown ### 4. Search, Visual Similarity & Opportunity Finder (Vector Algebra) Parameters
SEARCH_QUERY_ID = "distribution_warehouses"  # @param ["solar_rooftops", "distribution_warehouses", "shipping_containers", "Gantry_cranes", "tennis_courts", "swimming_pools", "parking_lots", "construction_sites", "water_treatment"]
SEED_TILE_ID = "r038_c052"  # @param {type:"string"}
POSITIVE_QUERY_ID = "distribution_warehouses"  # @param ["distribution_warehouses", "parking_lots", "shipping_containers", "construction_sites", "Gantry_cranes"]
NEGATIVE_QUERY_ID = "solar_rooftops"  # @param ["solar_rooftops", "swimming_pools", "tennis_courts", "water_treatment"]
NEGATIVE_WEIGHT = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.1}
TOP_K_RESULTS = 10  # @param {type:"slider", min:5, max:25, step:5}
VISUAL_GRID_TILES = 6  # @param {type:"slider", min:3, max:9, step:3}

# @markdown ---
# @markdown ### 5. Resource Cleanup Settings
UNDEPLOY_ENDPOINT_ON_FINISH = False  # @param {type:"boolean"}
DELETE_ENDPOINT_AND_MODEL = False  # @param {type:"boolean"}

# Resolve fallbacks from Colab secrets or Application Default Credentials
if not PROJECT_ID:
    PROJECT_ID = _COLAB_SECRET_PROJECT or os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if not GMP_API_KEY:
    GMP_API_KEY = _COLAB_SECRET_GMP_KEY or os.environ.get("GMP_API_KEY", "")

if not PROJECT_ID:
    raise ValueError(
        "Please set PROJECT_ID in Step 2 (Configuration) or configure GCP_PROJECT_ID in Colab Secrets."
    )

TILES_TABLE_FQN = f"{PROJECT_ID}.{BQ_DATASET}.{TILES_TABLE_NAME}"
QUERIES_TABLE_FQN = f"{PROJECT_ID}.{BQ_DATASET}.{QUERIES_TABLE_NAME}"

aiplatform.init(project=PROJECT_ID, location=REGION)
bq_client = bigquery.Client(project=PROJECT_ID)

selected_model_cfg = utils.MODEL_CONFIGS[MODEL_TYPE]
selected_platform_cfg = utils.PLATFORM_CONFIGS[ACCELERATOR]

config_summary_df = pd.DataFrame(
    [
        {"Setting": "GCP Project & Region", "Value": f"{PROJECT_ID} ({REGION})"},
        {"Setting": "BigQuery Tiles Table", "Value": TILES_TABLE_FQN},
        {"Setting": "BigQuery Queries Table", "Value": QUERIES_TABLE_FQN},
        {
            "Setting": "AOI & NAIP Tile Grid",
            "Value": f"{AOI_NAME} ({TILE_SIZE_PX}x{TILE_SIZE_PX} px @ 0.6m GSD, FETCH_LIVE_NAIP_TILES={FETCH_LIVE_NAIP_TILES})",
        },
        {
            "Setting": "Aerial & Satellite Model (`utils.MODEL_CONFIGS`)",
            "Value": f"{MODEL_TYPE} (RS-SigLIP2) -> id={selected_model_cfg.model_id}, mode={MODEL_MODE}",
        },
        {
            "Setting": "Platform Config (`utils.PLATFORM_CONFIGS`)",
            "Value": f"{ACCELERATOR} -> machine_type={selected_platform_cfg.machine_type}",
        },
        {
            "Setting": "Agent Platform Endpoint Mode",
            "Value": (
                f"Attach to {EXISTING_ENDPOINT_ID}"
                if EXISTING_ENDPOINT_ID
                else ("Deploy New Endpoint" if DEPLOY_LIVE_ENDPOINT else "Pre-computed BigQuery Mode (Fast)")
            ),
        },
        {
            "Setting": "Active Search & Algebra Config",
            "Value": f"Query='{SEARCH_QUERY_ID}' | Seed='{SEED_TILE_ID}' | Algebra='{POSITIVE_QUERY_ID} - {NEGATIVE_WEIGHT}*{NEGATIVE_QUERY_ID}'",
        },
    ]
)
print("✅ Configuration validated:")
display(config_summary_df)

## Step 3: Aerial and Satellite Models (`RS-SigLIP2`) on Agent Platform Model Garden

### About Aerial and Satellite Models & `RS-SigLIP2`
Google Maps Platform's **Aerial and Satellite Models** are part of the **Google Earth AI Remote Sensing Foundation Models** effort and are available in **Agent Platform Model Garden**. This notebook uses **`RS-SigLIP2`** (`SIGLIP2_512_RC1` / `SIGLIP2_224_RC1`), a two-tower vision-language foundation model trained on overhead aerial and satellite imagery using a Sigmoid Loss for Language-Image Pre-Training (SigLIP 2) objective.

### Using `remote_sensing.vertex_ai.utils` for Model Creation & Deployment
All model metadata, container URIs (`utils.SERVE_DOCKER_URI`), checkpoint paths (`utils.MODEL_CONFIGS`), and hardware profiles (`utils.PLATFORM_CONFIGS`) are managed by the official [`remote_sensing/vertex_ai/utils.py`](https://github.com/google-research/remote-sensing/blob/main/remote_sensing/vertex_ai/utils.py) module cloned in Step 1:

- **`utils.create_model(...)`**: Registers the selected Aerial and Satellite Model (`SIGLIP2_512_RC1`, `SIGLIP2_224_RC1`, or `MAMMUT`) with the official serving container (`us-docker.pkg.dev/vertex-ai-restricted/vertex-vision-model-garden-remote-sensing-dockers/remote-sensing-serve-tf-gpu:latest`) and environment variables (`MODEL_ID`, `MODEL_PATH`, `MODEL_TYPE`, `MODEL_MODE`, `PLATFORM`). Passing `batch_model=True` prepares the model for `model.batch_predict(...)`.
- **`utils.deploy_model(...)`**: Creates an Online Prediction Endpoint (`dedicated_endpoint_enabled=USE_DEDICATED_ENDPOINT`) and deploys the model onto the hardware profile from `utils.PLATFORM_CONFIGS[ACCELERATOR]` (e.g., `g2-standard-8` with `1× NVIDIA_L4`).
- **`utils.b64_png(image)`**: Encodes `PIL.Image` patches to base64 PNG strings for `{"image": utils.b64_png(img)}`, `{"text": prompt}`, or `{"image": utils.b64_png(img), "texts": [...]}` requests.

> **Best Practice — Why We Rank by Cosine Similarity (`1 - ML.DISTANCE`) Instead of Raw `similarity_scores`**:
> In `COMBINED` mode, `similarity_scores` applies an uncalibrated sigmoid (`1 / (1 + exp(-dot))`). Storing the L2-normalized `1152-D` `image_embedding` and `text_embedding` vectors in BigQuery and ranking by **cosine similarity (`1 - ML.DISTANCE(img, txt, 'COSINE')`)** plus **regional z-score (`+σ` above the metro median)** provides well-calibrated separation (`+2.5σ to +6.0σ`) across tens of thousands of tiles.

In [ ]:
# @title 3. Deploy/Attach Aerial and Satellite Model (`RS-SigLIP2`) via `remote_sensing.vertex_ai.utils`
endpoint: Optional[aiplatform.Endpoint] = None
model: Optional[aiplatform.Model] = None
unique_suffix = uuid.uuid4().hex[:6]

if EXISTING_ENDPOINT_ID:
    endpoint = aiplatform.Endpoint(EXISTING_ENDPOINT_ID)
    print(f"✅ Attached to existing Endpoint: {endpoint.resource_name}")
elif DEPLOY_LIVE_ENDPOINT:
    print(
        f"🚀 Creating & deploying {MODEL_TYPE} ({MODEL_MODE}) on {ACCELERATOR} "
        "via remote_sensing.vertex_ai.utils..."
    )
    model = utils.create_model(
        display_name=f"{DISPLAY_NAME}-{MODEL_TYPE.lower()}-{MODEL_MODE.lower()}-{ACCELERATOR.lower()}-{unique_suffix}",
        model_type=MODEL_TYPE,
        model_mode=MODEL_MODE,
        accelerator=ACCELERATOR,
        description=f"Model {DISPLAY_NAME} ({MODEL_TYPE}/{MODEL_MODE} on {ACCELERATOR})",
    )
    print(f"Created model: {model.resource_name} ({model.display_name})")

    endpoint = utils.deploy_model(
        endpoint_name=f"{DISPLAY_NAME}-{unique_suffix}-endpoint",
        model=model,
        accelerator=ACCELERATOR,
        service_account=SERVICE_ACCOUNT if SERVICE_ACCOUNT else None,
        min_replica_count=MIN_REPLICA_COUNT,
        max_replica_count=MAX_REPLICA_COUNT,
        use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
    )
    print(f"✅ Deployed live endpoint: {endpoint.resource_name}")
else:
    print(
        "ℹ️ DEPLOY_LIVE_ENDPOINT is False — skipping GPU deployment and using "
        f"pre-computed 1152-D RS-SigLIP2 embeddings in `{TILES_TABLE_FQN}` and `{QUERIES_TABLE_FQN}`."
    )

# Optional: Run live online inference using the official remote_sensing.vertex_ai.utils protocol
if endpoint is not None:
    sample_prompts = [
        "solar panels on warehouse roof",
        "shipping containers and intermodal freight yard",
    ]
    sample_img = Image.new("RGB", (TILE_SIZE_PX, TILE_SIZE_PX), color=(128, 136, 120))

    if MODEL_MODE in ("IMAGE_ONLY", "COMBINED"):
        img_result = endpoint.predict(
            instances=[{"image": utils.b64_png(sample_img)}],
            parameters={"batch_size": 1},
            use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
        )
        img_emb = img_result.predictions[0]["image_embedding"]
        print(f"  • Image embedding length: {len(img_emb)}, L2 norm: {np.linalg.norm(img_emb):.4f}")

    if MODEL_MODE in ("TEXT_ONLY", "COMBINED"):
        txt_result = endpoint.predict(
            instances=[{"text": prompt} for prompt in sample_prompts],
            parameters={"batch_size": len(sample_prompts)},
            use_dedicated_endpoint=USE_DEDICATED_ENDPOINT,
        )
        txt_emb = txt_result.predictions[0]["text_embedding"]
        print(f"  • Text embedding length: {len(txt_emb)}, L2 norm: {np.linalg.norm(txt_emb):.4f}")

# Optional: Demonstrate Batch Prediction via utils.create_model(..., batch_model=True)
if RUN_BATCH_JOB_EXAMPLE and GCS_BUCKET_NAME:
    batch_model = utils.create_model(
        display_name=f"{DISPLAY_NAME}-{MODEL_TYPE.lower()}-batch-{unique_suffix}",
        model_type=MODEL_TYPE,
        model_mode=MODEL_MODE,
        accelerator=ACCELERATOR,
        batch_model=True,
    )
    platform_config = utils.PLATFORM_CONFIGS[ACCELERATOR]
    batch_input_uri = f"gs://{GCS_BUCKET_NAME}/batch_inference/inputs/image*.jsonl"
    batch_output_uri = f"gs://{GCS_BUCKET_NAME}/batch_inference/outputs/"
    job_display_name = f"{DISPLAY_NAME}-batch-{unique_suffix}"
    print(f"📦 Launching BatchPredictionJob -> {batch_output_uri}")
    batch_job = batch_model.batch_predict(
        job_display_name=job_display_name,
        gcs_source=batch_input_uri,
        gcs_destination_prefix=batch_output_uri,
        instances_format=INPUT_SOURCE_FORMAT,
        machine_type=platform_config.machine_type,
        accelerator_count=platform_config.accelerator_count,
        accelerator_type=platform_config.accelerator_type,
        starting_replica_count=MIN_REPLICA_COUNT,
        max_replica_count=MAX_REPLICA_COUNT,
        labels={
            "task": "batch-inference",
            "vertex-ai-pipelines-run-billing-id": job_display_name,
        },
        batch_size=BATCH_SIZE,
        sync=False,
    )
    print(f"✅ Started BatchPredictionJob: {batch_job.resource_name}")

## Step 4: Inspect (or Populate) the BigQuery Spatial & `1152-D` Embedding Tables

Once the USDA NAIP regional tiles (`IMAGE_ONLY`) and domain search prompts (`TEXT_ONLY`) are encoded by the **`RS-SigLIP2`** Aerial and Satellite Model, they live in two BigQuery tables:

1. **`region_tiles_*` (`TILES_TABLE_FQN`)**:
   - `tile_id`, `row_idx`, `col_idx`: Grid coordinates across the metropolitan AOI.
   - `center_lat`, `center_lng`, `min_lat`, `min_lng`, `max_lat`, `max_lng`: Bounding box of each `307 m × 307 m` (`512×512 px` @ `0.6 m` GSD) tile.
   - `zone`: Sub-regional neighborhood label (e.g., *Port of LA / San Pedro*, *Port of Long Beach*, *LAX / El Segundo*).
   - `image_embedding`: `ARRAY<FLOAT64>` of length **`1,152`** ($\|v_{\text{img}}\|_2 = 1$).
2. **`region_queries_*` (`QUERIES_TABLE_FQN`)**:
   - `query_id`, `label`, `prompt`: Curated infrastructure, logistics, energy, and environmental text queries.
   - `text_embedding`: `ARRAY<FLOAT64>` of length **`1,152`** ($\|v_{\text{txt}}\|_2 = 1$).

If `FETCH_LIVE_NAIP_TILES = True` in Step 2, this step also demonstrates fetching public-domain `512×512` orthoimagery tiles directly from the **USGS National Map NAIP ImageServer** (`https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPPlus/ImageServer/exportImage`).

In [ ]:
# @title 4. Verify BigQuery Tile Coverage, Indexed Text Prompts & Optional Live USGS NAIP Tile Fetch
USGS_NAIP_EXPORT_URL = (
    "https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPPlus/ImageServer/exportImage"
)


def fetch_usgs_naip_tile(
    min_lng: float, min_lat: float, max_lng: float, max_lat: float, size_px: int = 512
) -> Image.Image:
    """Fetches a public-domain USDA/USGS NAIP orthoimagery tile (17 U.S.C. 105)."""
    params = {
        "bbox": f"{min_lng},{min_lat},{max_lng},{max_lat}",
        "bboxSR": "4326",
        "imageSR": "4326",
        "size": f"{size_px},{size_px}",
        "format": "jpgpng",
        "f": "image",
    }
    resp = requests.get(USGS_NAIP_EXPORT_URL, params=params, timeout=30)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")


if FETCH_LIVE_NAIP_TILES:
    print(f"🛰️ Fetching sample {TILE_SIZE_PX}x{TILE_SIZE_PX} public-domain USGS NAIP tile for {AOI_NAME}...")
    sample_naip_img = fetch_usgs_naip_tile(
        AOI_MIN_LNG, AOI_MIN_LAT, AOI_MIN_LNG + 0.0033, AOI_MIN_LAT + 0.0027, TILE_SIZE_PX
    )
    plt.figure(figsize=(4.5, 4.5))
    plt.imshow(sample_naip_img)
    plt.title(f"Sample USGS NAIP Tile ({AOI_NAME})", fontsize=11, fontweight="bold")
    plt.axis("off")
    plt.show()

zone_summary_sql = f"""
SELECT
  zone,
  COUNT(*) AS tile_count,
  ROUND(COUNT(*) * 0.3072 * 0.3072, 1) AS area_km2,
  ROUND(AVG(ARRAY_LENGTH(image_embedding)), 0) AS embedding_dims,
  ROUND(MIN(min_lat), 4) AS south_lat,
  ROUND(MAX(max_lat), 4) AS north_lat,
  ROUND(MIN(min_lng), 4) AS west_lng,
  ROUND(MAX(max_lng), 4) AS east_lng
FROM `{TILES_TABLE_FQN}`
GROUP BY zone
ORDER BY tile_count DESC
"""

zone_df = bq_client.query(zone_summary_sql).to_dataframe()
total_tiles = int(zone_df["tile_count"].sum())
total_km2 = float(zone_df["area_km2"].sum())
print(
    f"📦 `{TILES_TABLE_FQN}`: {total_tiles:,} USDA NAIP tiles (~{total_km2:,.1f} km²) across {len(zone_df)} zones:"
)
display(zone_df)

queries_sql = f"""
SELECT
  query_id,
  label,
  prompt,
  ARRAY_LENGTH(text_embedding) AS embedding_dims
FROM `{QUERIES_TABLE_FQN}`
ORDER BY query_id
"""
queries_df = bq_client.query(queries_sql).to_dataframe()
print(f"\n🔎 `{QUERIES_TABLE_FQN}`: {len(queries_df)} pre-encoded 1152-D text embeddings:")
display(queries_df)

## Step 5: Three Metro-Scale BigQuery Retrieval Patterns

Because both images and text share the same L2-normalized `1152-D` `RS-SigLIP2` vector space in BigQuery, we can execute three distinct geospatial intelligence patterns in pure SQL without spinning up a GPU at query time:

| Retrieval Mode | Input | BigQuery SQL Formula | Real-World Use Case |
| :--- | :--- | :--- | :--- |
| **5A. Text-to-Tile Semantic Search** | Text prompt vector $v_{\text{txt}}$ | `1.0 - ML.DISTANCE(t.image_embedding, q.text_embedding, 'COSINE')` | Find all distribution warehouses, gantry cranes, solar rooftops, or water treatment plants across a `1,650+ km²` metro region. |
| **5B. Tile-to-Tile Visual Similarity** | Seed tile vector $v_{\text{seed}}$ | `1.0 - ML.DISTANCE(t.image_embedding, s.image_embedding, 'COSINE')` | Click one known facility (e.g., an intermodal rail yard or terminal berth) and find every visually similar footprint across the metro grid. |
| **5C. Opportunity Finder (Vector Algebra)** | Positive prompt $v_{+}$ & Negative prompt $v_{-}$ | `score(v_+) - α * score(v_-)` (with regional z-score normalization) | **Solar Rooftop Prospecting**: Find large commercial warehouse roofs (`+distribution_warehouses`) that **do not yet** have solar arrays (`-0.5 * solar_rooftops`). |

Below, we execute all three retrieval patterns using the parameters configured in **Step 2**.

In [ ]:
# @title 5A. Text-to-Tile Semantic Search (`ML.DISTANCE` + Regional Z-Score) & Visual Tile Grid
text_search_sql = f"""
WITH raw_scores AS (
  SELECT
    t.tile_id,
    t.zone,
    t.row_idx,
    t.col_idx,
    ROUND(t.center_lat, 5) AS center_lat,
    ROUND(t.center_lng, 5) AS center_lng,
    t.gcs_uri,
    q.label AS query_label,
    q.prompt AS query_prompt,
    1.0 - ML.DISTANCE(t.image_embedding, q.text_embedding, 'COSINE') AS cosine_sim
  FROM `{TILES_TABLE_FQN}` AS t
  CROSS JOIN `{QUERIES_TABLE_FQN}` AS q
  WHERE q.query_id = @query_id
),
stats AS (
  SELECT
    AVG(cosine_sim) AS mean_sim,
    STDDEV(cosine_sim) AS std_sim
  FROM raw_scores
)
SELECT
  ROW_NUMBER() OVER (ORDER BY r.cosine_sim DESC) AS rank,
  r.tile_id,
  r.zone,
  r.center_lat,
  r.center_lng,
  ROUND(r.cosine_sim, 4) AS cosine_sim,
  ROUND(SAFE_DIVIDE(r.cosine_sim - s.mean_sim, s.std_sim), 2) AS z_score_sigma,
  r.gcs_uri,
  r.query_label,
  r.query_prompt
FROM raw_scores AS r
CROSS JOIN stats AS s
ORDER BY r.cosine_sim DESC
LIMIT @top_k
"""

job_config_5a = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("query_id", "STRING", SEARCH_QUERY_ID),
        bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
    ]
)
search_results_df = bq_client.query(text_search_sql, job_config=job_config_5a).to_dataframe()

query_title = search_results_df["query_label"].iloc[0] if not search_results_df.empty else SEARCH_QUERY_ID
query_prompt_str = search_results_df["query_prompt"].iloc[0] if not search_results_df.empty else ""
print(f"🔎 Mode 5A — Text-to-Tile Search for `{SEARCH_QUERY_ID}` ({query_title})")
print(f"   Prompt: '{query_prompt_str}'")
display(
    search_results_df[
        ["rank", "tile_id", "zone", "center_lat", "center_lng", "cosine_sim", "z_score_sigma"]
    ]
)


def plot_naip_tiles_from_gcs(df: pd.DataFrame, title: str, max_tiles: int = 6) -> None:
    """Renders USDA NAIP public-domain tiles from GCS (or live USGS fallback) if accessible."""
    subset = df.head(max_tiles)
    n = len(subset)
    if n == 0:
        return
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
    axes_flat = np.array(axes).reshape(-1) if n > 1 else [axes]
    storage_client = storage.Client(project=PROJECT_ID)

    for idx, (_, row) in enumerate(subset.iterrows()):
        ax = axes_flat[idx]
        gcs_uri = str(row.get("gcs_uri", ""))
        img = None
        try:
            if gcs_uri.startswith("gs://"):
                parts = gcs_uri[5:].split("/", 1)
                bucket = storage_client.bucket(parts[0])
                blob = bucket.blob(parts[1])
                img_bytes = blob.download_as_bytes()
                img = Image.open(io.BytesIO(img_bytes))
        except Exception:
            img = None
        if img is None:
            try:
                c_lat, c_lng = float(row["center_lat"]), float(row["center_lng"])
                img = fetch_usgs_naip_tile(
                    c_lng - 0.00168, c_lat - 0.00138, c_lng + 0.00168, c_lat + 0.00138, 256
                )
            except Exception:
                img = None
        if img is not None:
            ax.imshow(img)
        else:
            ax.text(
                0.5,
                0.5,
                f"Tile {row['tile_id']}\n({row['zone']})\n({row['center_lat']}, {row['center_lng']})",
                ha="center",
                va="center",
                fontsize=10,
            )
            ax.set_facecolor("#F1F3F4")
        score_label = (
            f"cos={row['cosine_sim']:.3f} (+{row['z_score_sigma']:.1f}σ)"
            if "cosine_sim" in row
            else f"algebra={row['net_opportunity_sigma']:+.2f}σ"
        )
        ax.set_title(f"#{int(row['rank'])} {row['tile_id']} | {score_label}", fontsize=10, fontweight="bold")
        ax.set_xlabel(f"{row['zone']}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for extra_ax in axes_flat[n:]:
        extra_ax.axis("off")
    plt.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


plot_naip_tiles_from_gcs(
    search_results_df,
    title=f"Top {min(int(VISUAL_GRID_TILES), len(search_results_df))} USDA NAIP Matches — {query_title}",
    max_tiles=int(VISUAL_GRID_TILES),
)

In [ ]:
# @title 5B. Tile-to-Tile Visual Similarity (`SEED_TILE_ID` Nearest Neighbors)
# If SEED_TILE_ID is not present in the active table, automatically fall back to Rank #1 from Step 5A
effective_seed_tile = (
    SEED_TILE_ID
    if not search_results_df.empty
    else "r038_c052"
)

tile_similarity_sql = f"""
WITH seed AS (
  SELECT
    tile_id AS seed_id,
    zone AS seed_zone,
    image_embedding AS seed_embedding
  FROM `{TILES_TABLE_FQN}`
  WHERE tile_id = @seed_tile_id
  LIMIT 1
)
SELECT
  ROW_NUMBER() OVER (
    ORDER BY 1.0 - ML.DISTANCE(t.image_embedding, s.seed_embedding, 'COSINE') DESC
  ) AS rank,
  t.tile_id,
  t.zone,
  ROUND(t.center_lat, 5) AS center_lat,
  ROUND(t.center_lng, 5) AS center_lng,
  ROUND(1.0 - ML.DISTANCE(t.image_embedding, s.seed_embedding, 'COSINE'), 4) AS cosine_sim,
  t.gcs_uri
FROM `{TILES_TABLE_FQN}` AS t
CROSS JOIN seed AS s
WHERE t.tile_id != s.seed_id
ORDER BY cosine_sim DESC
LIMIT @top_k
"""

job_config_5b = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("seed_tile_id", "STRING", effective_seed_tile),
        bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
    ]
)
similar_tiles_df = bq_client.query(tile_similarity_sql, job_config=job_config_5b).to_dataframe()

if similar_tiles_df.empty and not search_results_df.empty:
    effective_seed_tile = str(search_results_df["tile_id"].iloc[0])
    job_config_5b = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("seed_tile_id", "STRING", effective_seed_tile),
            bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
        ]
    )
    similar_tiles_df = bq_client.query(tile_similarity_sql, job_config=job_config_5b).to_dataframe()

print(f"🧭 Mode 5B — Top {len(similar_tiles_df)} Visually Similar Tiles to Seed Tile `{effective_seed_tile}`:")
display(similar_tiles_df[["rank", "tile_id", "zone", "center_lat", "center_lng", "cosine_sim"]])

In [ ]:
# @title 5C. Opportunity Finder — Vector Algebra (`POSITIVE_QUERY_ID - NEGATIVE_WEIGHT * NEGATIVE_QUERY_ID`)
vector_algebra_sql = f"""
WITH pos_q AS (
  SELECT label, text_embedding FROM `{QUERIES_TABLE_FQN}` WHERE query_id = @pos_id
),
neg_q AS (
  SELECT label, text_embedding FROM `{QUERIES_TABLE_FQN}` WHERE query_id = @neg_id
),
tile_scores AS (
  SELECT
    t.tile_id,
    t.zone,
    ROUND(t.center_lat, 5) AS center_lat,
    ROUND(t.center_lng, 5) AS center_lng,
    t.gcs_uri,
    p.label AS pos_label,
    n.label AS neg_label,
    1.0 - ML.DISTANCE(t.image_embedding, p.text_embedding, 'COSINE') AS pos_cos,
    1.0 - ML.DISTANCE(t.image_embedding, n.text_embedding, 'COSINE') AS neg_cos
  FROM `{TILES_TABLE_FQN}` AS t
  CROSS JOIN pos_q AS p
  CROSS JOIN neg_q AS n
),
stats AS (
  SELECT
    AVG(pos_cos) AS mean_pos, STDDEV(pos_cos) AS std_pos,
    AVG(neg_cos) AS mean_neg, STDDEV(neg_cos) AS std_neg
  FROM tile_scores
),
normalized AS (
  SELECT
    ts.*,
    SAFE_DIVIDE(ts.pos_cos - s.mean_pos, s.std_pos) AS pos_z,
    SAFE_DIVIDE(ts.neg_cos - s.mean_neg, s.std_neg) AS neg_z,
    SAFE_DIVIDE(ts.pos_cos - s.mean_pos, s.std_pos)
      - (@neg_weight * SAFE_DIVIDE(ts.neg_cos - s.mean_neg, s.std_neg)) AS net_opportunity_sigma
  FROM tile_scores AS ts
  CROSS JOIN stats AS s
)
SELECT
  ROW_NUMBER() OVER (ORDER BY net_opportunity_sigma DESC) AS rank,
  tile_id,
  zone,
  center_lat,
  center_lng,
  ROUND(pos_cos, 4) AS pos_cosine,
  ROUND(pos_z, 2) AS pos_sigma,
  ROUND(neg_cos, 4) AS neg_cosine,
  ROUND(neg_z, 2) AS neg_sigma,
  ROUND(net_opportunity_sigma, 2) AS net_opportunity_sigma,
  gcs_uri
FROM normalized
ORDER BY net_opportunity_sigma DESC
LIMIT @top_k
"""

job_config_5c = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("pos_id", "STRING", POSITIVE_QUERY_ID),
        bigquery.ScalarQueryParameter("neg_id", "STRING", NEGATIVE_QUERY_ID),
        bigquery.ScalarQueryParameter("neg_weight", "FLOAT64", float(NEGATIVE_WEIGHT)),
        bigquery.ScalarQueryParameter("top_k", "INT64", int(TOP_K_RESULTS)),
    ]
)
algebra_df = bq_client.query(vector_algebra_sql, job_config=job_config_5c).to_dataframe()

print(
    f"⚡ Mode 5C — Opportunity Finder Vector Algebra: "
    f"`+{POSITIVE_QUERY_ID}` minus `{NEGATIVE_WEIGHT:.1f} × {NEGATIVE_QUERY_ID}`"
)
display(
    algebra_df[
        [
            "rank",
            "tile_id",
            "zone",
            "center_lat",
            "center_lng",
            "pos_sigma",
            "neg_sigma",
            "net_opportunity_sigma",
        ]
    ]
)

## Step 6: Interactive Map Visualization (Google Maps 2D Tiles + Bounding Boxes)

Next, let's render the top overhead matches on an interactive **Folium** map so you can visually inspect each `307 m × 307 m` tile footprint:
- **Red (`#D93025`) Bounding Boxes**: Top **Text-to-Tile (`SEARCH_QUERY_ID`)** semantic search matches from Step 5A.
- **Green (`#1E8E3E`) Dashed Bounding Boxes**: Top **Opportunity Finder (`POSITIVE_QUERY_ID - NEGATIVE_WEIGHT * NEGATIVE_QUERY_ID`)** vector algebra matches from Step 5C.
- **Basemap Layer**: Uses the official **Google Maps 2D Hybrid/Satellite Tiles** overlay when `GMP_API_KEY` is provided in Step 2 (or standard CartoDB Positron basemaps when `GMP_API_KEY` is blank).

In [ ]:
# @title 6. Render Interactive Folium Map of Top Matches
if not search_results_df.empty:
    map_center_lat = float(search_results_df["center_lat"].mean())
    map_center_lng = float(search_results_df["center_lng"].mean())
    half_deg_lat = 0.00138  # ~153.6m half-tile height
    half_deg_lng = 0.00168  # ~153.6m half-tile width at ~34 deg latitude

    m = folium.Map(
        location=[map_center_lat, map_center_lng],
        zoom_start=11,
        tiles="CartoDB positron",
        control_scale=True,
    )

    if GMP_API_KEY:
        folium.TileLayer(
            tiles=f"https://mt1.google.com/vt/lyrs=y&x={{x}}&y={{y}}&z={{z}}&key={GMP_API_KEY}",
            attr="Google Maps",
            name="Google Maps 2D Hybrid",
            overlay=False,
            control=True,
        ).add_to(m)

    # Layer 1: Step 5A Text-to-Tile Search Matches
    search_layer = folium.FeatureGroup(name=f"5A: Text Search ({SEARCH_QUERY_ID})", show=True)
    for _, row in search_results_df.iterrows():
        c_lat, c_lng = float(row["center_lat"]), float(row["center_lng"])
        bounds = [
            [c_lat - half_deg_lat, c_lng - half_deg_lng],
            [c_lat + half_deg_lat, c_lng + half_deg_lng],
        ]
        popup_html = (
            f"<b>#{int(row['rank'])} Tile {row['tile_id']}</b><br>"
            f"Zone: {row['zone']}<br>"
            f"Query: <code>{SEARCH_QUERY_ID}</code><br>"
            f"Cosine Sim: <b>{row['cosine_sim']:.4f}</b> (+{row['z_score_sigma']:.2f}σ)"
        )
        folium.Rectangle(
            bounds=bounds,
            color="#D93025",
            weight=2,
            fill=True,
            fill_color="#D93025",
            fill_opacity=0.28,
            popup=folium.Popup(popup_html, max_width=280),
            tooltip=f"#{int(row['rank'])} {row['tile_id']} ({row['zone']}) | +{row['z_score_sigma']:.2f}σ",
        ).add_to(search_layer)
    search_layer.add_to(m)

    # Layer 2: Step 5C Opportunity Finder Vector Algebra Matches
    if not algebra_df.empty:
        algebra_layer = folium.FeatureGroup(
            name=f"5C: Opportunity Finder (+{POSITIVE_QUERY_ID} - {NEGATIVE_WEIGHT}*{NEGATIVE_QUERY_ID})",
            show=True,
        )
        for _, row in algebra_df.iterrows():
            c_lat, c_lng = float(row["center_lat"]), float(row["center_lng"])
            bounds = [
                [c_lat - half_deg_lat, c_lng - half_deg_lng],
                [c_lat + half_deg_lat, c_lng + half_deg_lng],
            ]
            popup_html = (
                f"<b>Opportunity #{int(row['rank'])}: {row['tile_id']}</b><br>"
                f"Zone: {row['zone']}<br>"
                f"Net Opportunity: <b>{row['net_opportunity_sigma']:+.2f}σ</b><br>"
                f"(+{POSITIVE_QUERY_ID}: {row['pos_sigma']:+.2f}σ | -{NEGATIVE_QUERY_ID}: {row['neg_sigma']:+.2f}σ)"
            )
            folium.Rectangle(
                bounds=bounds,
                color="#1E8E3E",
                weight=2,
                dash_array="5, 5",
                fill=True,
                fill_color="#1E8E3E",
                fill_opacity=0.22,
                popup=folium.Popup(popup_html, max_width=300),
                tooltip=f"Opportunity #{int(row['rank'])} {row['tile_id']} | {row['net_opportunity_sigma']:+.2f}σ",
            ).add_to(algebra_layer)
        algebra_layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    display(m)

## Step 7: Optional Resource Cleanup

If you deployed a dedicated Online Prediction Endpoint in **Step 3** (`DEPLOY_LIVE_ENDPOINT = True`), undeploying or deleting the endpoint when your session finishes avoids idle GPU billing (`NVIDIA_L4` / `A100`). Control this behavior using `UNDEPLOY_ENDPOINT_ON_FINISH` and `DELETE_ENDPOINT_AND_MODEL` in **Step 2**.

In [ ]:
# @title 7. Undeploy or Delete Endpoint (Controlled via Step 2 Parameters)
if endpoint is not None and UNDEPLOY_ENDPOINT_ON_FINISH:
    if DELETE_ENDPOINT_AND_MODEL:
        print(f"🧹 Deleting endpoint {endpoint.resource_name} (force=True) and model...")
        endpoint.delete(force=True)
        if model is not None:
            model.delete()
        print("✅ Deleted Endpoint and Model resource.")
    else:
        print(f"🧹 Undeploying all models from {endpoint.resource_name}...")
        endpoint.undeploy_all()
        print("✅ Undeployed GPU replicas (Endpoint metadata retained).")
else:
    print("ℹ️ No Endpoint teardown requested (or running in Pre-computed BigQuery Mode).")